In [ ]:
import warnings
warnings.filterwarnings('ignore') 
import scanpy as sc
import anndata as ad
import gseapy as gp
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from scipy import sparse

import SILPAG as sp

In [ ]:
# data preprocess
ref = sc.read_h5ad('/data/luy/SILPAG/data/ST_human_BreastCancer/V02_nb.h5ad')
adata = sc.read_h5ad('/data/luy/SILPAG/data/ST_human_BreastCancer/H1_nb.h5ad')
ref.var_names_make_unique()
adata.var_names_make_unique()
sp.prefilter_specialgenes(ref)
sp.prefilter_specialgenes(adata)

gene = list(set(ref.var_names)&set(adata.var_names))
ref = ref[:,gene]
adata = adata[:, gene]

g1 = sc.pp.filter_genes(ref, min_cells=int(ref.shape[0]*0.02), inplace=False)
g2 = sc.pp.filter_genes(adata, min_cells=int(adata.shape[0]*0.02), inplace=False)
ref = ref[:,g1[0] | g2[0]]
adata = adata[:, g1[0] | g2[0]]

print(ref.X.max(), adata.X.max())
print(ref.shape, adata.shape)

In [ ]:
# convert to grey-scale images
dataset1, indicator1, _ = sp.data_process(ref)
dataset2, indicator2, _ = sp.data_process(adata)
dataset1[dataset1 > 100] = 100
dataset2[dataset2 > 100] = 100
ref.varm['gene_img'] = dataset1
adata.varm['gene_img'] = dataset2

In [ ]:
# load housekeeping genes
hkgene = pd.read_csv('/data/luy/SILPAG/data/Housekeeping_GenesHuman.csv', sep=';')
hkgene = list(hkgene['Gene.name'])
adata.var['hkgene'] = [1 if i in hkgene else 0 for i in adata.var_names]
ref.var['hkgene'] = [1 if i in hkgene else 0 for i in ref.var_names]
labels = np.zeros(ref.shape[1], dtype=int)
labels[np.where(ref.var['hkgene'] == 1)[0]] = 1

In [ ]:
args = sp.Config()
args = args.parse()
args.device = 'cuda:0'
args.distr = ['nb', 'nb']# ['gaussian', 'gaussian']
args.K = [1024] * args.num_slice
args.patch_size = [(4, 4), (4, 4)]
args.resize_factor = [1, 1]
args.hist = [False, False]
args.beta = 1e3 # cross-view loss weight
args.learning_rate = 1.5e-3
args.weight_decay = 5e-2 # L2 regularization
args.batch_size = 64
args.anchor_size = 512
args.pre_epochs = 30
args.warmup_epochs = 20
args.epochs = 100
args.anchor_epochs = 10
args.lr_decay_epochs = [200]# [120,160]
args.trace = True
args.save = True
args.model_path = '/data/luy/SILPAG/saved_model/hBC_nb'
args.train = True

gene_images = [ref.varm['gene_img'], adata.varm['gene_img']]
train_dataset = sp.GeneDataset(gene_images, labels, args)

In [ ]:
# train model
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, drop_last=False)
model, dataset, PI, Q = sp.train_marker(train_loader, hist_data=None, args=args)
print('Number of Anchor gene:', dataset.labels.sum())
code = sp.get_code(model, dataset, hist_data=None, pi_init=PI, source_idx=0, target_idx=1, args=args)
adata.varm['code_ref'] = code[0]
adata.varm['code_tgt'] = code[1]

In [ ]:
# view-transferred generation
result = model.generate(args, adata, tgt_index=1, code_id='code_ref', gene='all') # use code_ref to generate targte SGMs